# Chapter 4: Surfaces in three dimensions

Source orientation: Andrew Pressley, *Elementary Differential Geometry*, Chapter 4, printed pages 67-94, PDF pages 75-101, sections 4.1-4.5. The source was used only to orient definitions, examples, and proof goals. This notebook gives original prose, code, diagrams, and checks.

## Chapter question

How can a subset of `R^3` be treated as a smooth surface when no single coordinate grid covers it, and what data survives a change of patch?

The chapter's answer has three layers. First, a surface is described locally by homeomorphic patches whose images cover it. Second, calculus is allowed only when patch changes are smooth and regular. Third, tangent planes, derivatives, and normals are defined through patches but must be checked against reparametrization. The Mobius band shows the final obstruction: tangent planes exist locally, but a coherent global normal may not.

## Translation guide

| Book-side idea | Computational translation |
| --- | --- |
| Surface patch `sigma: U -> R^3` | A Python function from two parameters to three coordinates, plus a declared open domain. |
| Atlas | A list of patches whose images cover the surface; seams are part of the data, not drawing defects. |
| Transition map `Phi = sigma^{-1} o sigma_tilde` | A coordinate conversion on overlaps; smoothness is tested by formulas and Jacobians. |
| Regular patch | The cross product `sigma_u x sigma_v` never vanishes, so parameter curves span a plane. |
| Smooth map between surfaces | A map whose coordinate representative between patches is smooth. |
| Derivative `D_p f` | The Jacobian of the coordinate representative, interpreted between tangent-plane bases. |
| Standard unit normal | The normalized cross product of the parameter-curve tangents. |
| Orientability | An atlas can be chosen so every transition has positive Jacobian determinant; otherwise normals must flip somewhere. |

## Route

1. Build the cylinder atlas and read its overlap as a coordinate change.
2. Check regularity and reparametrization by the cross-product identity.
3. Use the plane-to-cylinder wrapping map as a smooth local diffeomorphism that is not global.
4. Draw a tangent plane from parameter curves and verify the chain rule.
5. Compare standard normals with oriented angle on the sphere.
6. Track the Mobius median normal and transition determinant signs to see the orientability obstruction.

In [ ]:
from pathlib import Path
import json
import math
import sys

start = Path.cwd().resolve()

def is_book_root(candidate: Path) -> bool:
    return (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists()

candidates = [start, *start.parents]
if start.is_dir():
    candidates.extend([child for child in start.iterdir() if child.is_dir()])

for candidate in candidates:
    if is_book_root(candidate):
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the Pressley book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

UNIT = "chapter-04"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts"
UNIT_ARTIFACT_ROOT = ARTIFACT_ROOT / UNIT
FIGURES_DIR = UNIT_ARTIFACT_ROOT / "figures"
INTERACTIVE_DIR = UNIT_ARTIFACT_ROOT / "interactive"
CHECKS_DIR = UNIT_ARTIFACT_ROOT / "checks"
TABLES_DIR = UNIT_ARTIFACT_ROOT / "tables"
for directory in (FIGURES_DIR, INTERACTIVE_DIR, CHECKS_DIR, TABLES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sympy as sp

from utils.artifacts import assert_artifact, display_artifact
from utils.plotting import PALETTE, require_nonblank_png, set_defaults

set_defaults()
ARTIFACT_PATHS = []
CHECKS = {}

def book_relative(path: Path) -> str:
    return Path(path).resolve().relative_to(BOOK_ROOT).as_posix()

def register(path: Path) -> Path:
    path = Path(path)
    if path not in ARTIFACT_PATHS:
        ARTIFACT_PATHS.append(path)
    return path

def save_png(fig, filename: str) -> Path:
    path = FIGURES_DIR / filename
    fig.savefig(path, dpi=170, bbox_inches="tight")
    plt.close(fig)
    register(path)
    CHECKS.setdefault("png_stats", {})[filename] = require_nonblank_png(path)
    return path

def save_plotly(fig: go.Figure, filename: str) -> Path:
    path = INTERACTIVE_DIR / filename
    fig.write_html(path, include_plotlyjs="cdn", full_html=True)
    register(path)
    return path

def save_check(data: dict, filename: str) -> Path:
    path = CHECKS_DIR / filename
    path.write_text(json.dumps(data, indent=2, sort_keys=True), encoding="utf-8")
    register(path)
    return path

def segment_trace(p, q, name, color, width=7, dash=None):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    line = {"color": color, "width": width}
    if dash:
        line["dash"] = dash
    return go.Scatter3d(
        x=[p[0], q[0]], y=[p[1], q[1]], z=[p[2], q[2]],
        mode="lines", line=line, name=name,
    )

def cone_trace(p, v, name, color, sizeref=0.25):
    p = np.asarray(p, dtype=float)
    v = np.asarray(v, dtype=float)
    return go.Cone(
        x=[p[0]], y=[p[1]], z=[p[2]],
        u=[v[0]], v=[v[1]], w=[v[2]],
        anchor="tail", sizemode="absolute", sizeref=sizeref,
        colorscale=[[0, color], [1, color]], showscale=False, name=name,
    )

## Implementation storyboard and library routing

| Concept | Representation | Library choice | Inspection target | Check |
| --- | --- | --- | --- | --- |
| Cylinder atlas and seams | Static interval diagram plus rotatable cylinder | Matplotlib for the proof-state diagram; Plotly for 3D rotation | The two angle cuts and the overlap branches | Sampled coverage and transition determinants |
| Smooth reparametrization | Commutative diagram and symbolic cross-product identity | Matplotlib plus SymPy | The determinant multiplying `sigma_u x sigma_v` | Exact zero residual |
| Smooth map and local diffeomorphism | Plane strip wrapping around a cylinder | Plotly plus a small Jacobian table | Same image for different plane points, but local Jacobian is invertible | Local determinant and non-injectivity witness |
| Tangent plane | Surface, parameter curves, tangent basis, normal arrow | Plotly for 3D inspection | `sigma_u`, `sigma_v`, and their span at one point | Chain-rule residual and orthogonality |
| Normals and oriented angle | Sphere normal field and signed angle computation | Plotly plus NumPy | Inward normal from the Pressley latitude-longitude patch | Unit length, tangent dot products, angle sign |
| Mobius obstruction | Twisted strip with transported median normals | Plotly plus symbolic transition branches | Normal returns with opposite sign after one circuit | Transition determinant signs and endpoint normal dot product |

In [ ]:
source_span = {
    "book": "Andrew Pressley, Elementary Differential Geometry, Second Edition",
    "chapter": "Chapter 4: Surfaces in three dimensions",
    "printed_pages": "67-94",
    "pdf_pages": "75-101",
    "sections": "4.1-4.5",
    "orientation_notes": [
        "4.1 defines surfaces by local patches and atlases, with cylinder, sphere, and cone failure mode examples.",
        "4.2 adds smoothness, regularity, transition maps, and reparametrization invariance.",
        "4.3 defines smooth maps, diffeomorphisms, and local diffeomorphisms between surfaces.",
        "4.4 defines tangent planes through curves and derivatives through pushed-forward velocities.",
        "4.5 defines standard normals, orientability, oriented angles, and the Mobius-band obstruction.",
    ],
}

visual_storyboard = {
    "chapter_goal": "Make patch-based surface calculus inspectable and coordinate-independent, then expose why the Mobius band has no global normal.",
    "visuals": [
        "cylinder atlas overlap and seams",
        "reparametrization determinant proof state",
        "plane-to-cylinder local diffeomorphism",
        "tangent plane from parameter curves",
        "sphere standard normal and oriented angle",
        "Mobius median normal flip and transition determinant signs",
    ],
    "libraries": {
        "matplotlib": "durable 2D proof-state and atlas diagrams",
        "plotly": "rotatable 3D surfaces, tangent planes, normal fields, and wrapping maps",
        "sympy": "exact Jacobian and cross-product identities",
        "numpy_pandas": "numeric residuals and regularity lab tables",
    },
}

save_check(source_span, "source-span.json")
save_check(visual_storyboard, "visual-storyboard.json")
source_span

## Surface patches and atlases: the seam is part of the data

A patch is not just a formula. It is a formula on an open parameter domain, and it must be one-to-one onto the surface piece it describes. The usual cylinder formula `(cos u, sin u, v)` becomes periodic if all real `u` are allowed, so a cylinder atlas uses angle intervals with different cuts. The seam lines below are not defects in the cylinder; they are where one chart stops being the preferred coordinate view and another chart takes over.

Inspect the interval diagram first: the overlap has two connected branches, and each branch has a simple smooth transition formula. Then rotate the cylinder artifact and compare the two vertical seam lines.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1.35, 1]})
ax = axes[0]
ax.set_title("Cylinder angle charts and overlap branches", color=PALETTE["ink"])
ax.set_xlim(-math.pi - 0.35, 2 * math.pi + 0.35)
ax.set_ylim(-0.3, 2.2)
ax.set_yticks([0.55, 1.45], ["U_tilde", "U"])
ax.set_xlabel("angle coordinate")
for spine in ax.spines.values():
    spine.set_color("#b6c0ca")
ax.grid(True, axis="x", color="#d7dde5", linewidth=0.7)
ax.plot([0, 2 * math.pi], [1.45, 1.45], color=PALETTE["blue"], linewidth=8, solid_capstyle="butt", label="U: 0 < u < 2*pi")
ax.plot([-math.pi, math.pi], [0.55, 0.55], color=PALETTE["gold"], linewidth=8, solid_capstyle="butt", label="U_tilde: -pi < u < pi")
for x, label in [(0, "0"), (math.pi, "pi"), (2 * math.pi, "2*pi"), (-math.pi, "-pi")]:
    ax.axvline(x, color="#9aa4b2", linestyle=":" if x not in (0, math.pi) else "--", linewidth=1)
    ax.text(x, -0.08, label, ha="center", va="top", fontsize=8)
ax.annotate("branch A: u = u_tilde", xy=(math.pi / 2, 1.0), xytext=(0.5, 1.85), arrowprops={"arrowstyle": "->", "color": PALETTE["teal"]}, color=PALETTE["teal"])
ax.annotate("branch B: u = u_tilde + 2*pi", xy=(-math.pi / 2, 1.0), xytext=(-2.9, 1.85), arrowprops={"arrowstyle": "->", "color": PALETTE["red"]}, color=PALETTE["red"])
ax.legend(loc="upper right", fontsize=8)

ax2 = axes[1]
theta = np.linspace(0, 2 * math.pi, 300)
ax2.plot(np.cos(theta), np.sin(theta), color=PALETTE["gray"], linewidth=2)
for th, color, label in [(0, PALETTE["red"], "U cut"), (math.pi, PALETTE["gold"], "U_tilde cut")]:
    ax2.plot([0, math.cos(th)], [0, math.sin(th)], color=color, linewidth=2)
    ax2.scatter([math.cos(th)], [math.sin(th)], color=color, s=45, label=label)
ax2.set_title("Cuts seen from above", color=PALETTE["ink"])
ax2.set_aspect("equal", adjustable="box")
ax2.set_xticks([])
ax2.set_yticks([])
ax2.legend(loc="lower left", fontsize=8)
for spine in ax2.spines.values():
    spine.set_color("#b6c0ca")
cylinder_atlas_png = save_png(fig, "cylinder-atlas-overlap-transition-map.png")

z = np.linspace(-1.4, 1.4, 50)
angle = np.linspace(0, 2 * math.pi, 120)
T, Z = np.meshgrid(angle, z, indexing="ij")
X, Y = np.cos(T), np.sin(T)
fig3d = go.Figure()
fig3d.add_surface(x=X, y=Y, z=Z, opacity=0.78, colorscale="Blues", showscale=False, name="unit cylinder")
for th, color, name in [(0, "#c44e52", "cut for U"), (math.pi, "#d59f0f", "cut for U_tilde")]:
    fig3d.add_trace(go.Scatter3d(
        x=np.full_like(z, math.cos(th)), y=np.full_like(z, math.sin(th)), z=z,
        mode="lines", line={"color": color, "width": 8}, name=name,
    ))
fig3d.update_layout(
    title="Cylinder atlas: two angle cuts cover one surface",
    scene={"aspectmode": "data", "xaxis_title": "x", "yaxis_title": "y", "zaxis_title": "z"},
    margin={"l": 0, "r": 0, "t": 45, "b": 0},
)
cylinder_atlas_html = save_plotly(fig3d, "cylinder-atlas-two-angle-charts.html")

sample_angles = np.array([0.0, 0.2, math.pi - 0.2, math.pi, math.pi + 0.2, 2 * math.pi - 0.2])
def in_U(theta_value):
    theta_value = theta_value % (2 * math.pi)
    return 0 < theta_value < 2 * math.pi

def in_U_tilde(theta_value):
    theta_value = ((theta_value + math.pi) % (2 * math.pi)) - math.pi
    return -math.pi < theta_value < math.pi

coverage = []
for th in sample_angles:
    charts = []
    if in_U(th):
        charts.append("U")
    if in_U_tilde(th):
        charts.append("U_tilde")
    coverage.append({"theta": float(th), "covering_charts": charts, "count": len(charts)})

transition_checks = {
    "cylinder_transition_branches": [
        {"domain": "0 < u_tilde < pi", "Phi": "(u_tilde, v_tilde)", "determinant": 1},
        {"domain": "-pi < u_tilde < 0", "Phi": "(u_tilde + 2*pi, v_tilde)", "determinant": 1},
    ],
    "sample_coverage": coverage,
    "minimum_sample_coverage": int(min(row["count"] for row in coverage)),
}
CHECKS["cylinder_atlas"] = transition_checks
save_check(transition_checks, "atlas-transition-checks.json")

display_artifact(cylinder_atlas_png, width=920)
display_artifact(cylinder_atlas_html, width="100%", height=520)

## Smooth transition maps: regularity is invariant data

Smooth compatibility is the rule that lets calculus on one chart agree with calculus on another. The key calculation is the one behind reparametrization: if `sigma_tilde = sigma o Phi`, then the new cross product is the old cross product multiplied by `det J(Phi)`. A nonzero determinant preserves regularity; a positive determinant also preserves the chosen normal direction.

The next cell checks this exactly for a saddle graph and a linear reparametrization. It also records a sphere graph-patch transition to show how smooth square-root formulas stay valid only on the overlap where their denominators are nonzero.

In [ ]:
u, v, a, b = sp.symbols("u v a b", real=True)
sigma = sp.Matrix([u, v, u**2 - v**2])
Phi = sp.Matrix([a + b, a - b])
J_Phi = Phi.jacobian([a, b])
det_Phi = sp.simplify(J_Phi.det())
sigma_cross = sigma.diff(u).cross(sigma.diff(v))
sigma_tilde = sigma.subs({u: Phi[0], v: Phi[1]})
tilde_cross = sigma_tilde.diff(a).cross(sigma_tilde.diff(b))
residual = sp.simplify(tilde_cross - det_Phi * sigma_cross.subs({u: Phi[0], v: Phi[1]}))

xcoord, ycoord = sp.symbols("xcoord ycoord", real=True)
z_plus_to_x_plus = sp.Matrix([ycoord, sp.sqrt(1 - xcoord**2 - ycoord**2)])
sphere_transition_det = sp.simplify(z_plus_to_x_plus.jacobian([xcoord, ycoord]).det())

fig, ax = plt.subplots(figsize=(8.8, 4.2))
ax.axis("off")
boxes = {"U_tilde": (0.10, 0.56), "U": (0.43, 0.56), "surface": (0.76, 0.56), "cross": (0.43, 0.14)}
for label, (x0, y0) in boxes.items():
    ax.add_patch(plt.Rectangle((x0 - 0.11, y0 - 0.075), 0.22, 0.15, facecolor="white", edgecolor="#9aa4b2", linewidth=1.4))
    ax.text(x0, y0, label, ha="center", va="center", color=PALETTE["ink"], fontsize=10)
ax.annotate("Phi", xy=boxes["U"], xytext=boxes["U_tilde"], arrowprops={"arrowstyle": "->", "lw": 1.8, "color": PALETTE["teal"]}, ha="center")
ax.annotate("sigma", xy=boxes["surface"], xytext=boxes["U"], arrowprops={"arrowstyle": "->", "lw": 1.8, "color": PALETTE["blue"]}, ha="center")
ax.annotate("sigma o Phi", xy=boxes["surface"], xytext=boxes["U_tilde"], arrowprops={"arrowstyle": "->", "lw": 1.2, "linestyle": "--", "color": PALETTE["gray"]}, ha="center")
ax.annotate("chain rule", xy=boxes["cross"], xytext=boxes["U"], arrowprops={"arrowstyle": "->", "lw": 1.2, "color": PALETTE["violet"]}, ha="center")
ax.text(0.43, 0.02, "(sigma o Phi)_a x (sigma o Phi)_b = det J(Phi) (sigma_u x sigma_v)", ha="center", color=PALETTE["ink"], fontsize=10)
ax.text(0.43, 0.30, f"for Phi(a,b)=(a+b,a-b), det J(Phi) = {det_Phi}", ha="center", color=PALETTE["red"], fontsize=10)
reparam_png = save_png(fig, "reparametrization-cross-product-proof-state.png")

reparam_check = {
    "Phi": ["a + b", "a - b"],
    "det_J_Phi": int(det_Phi),
    "symbolic_residual": [str(entry) for entry in residual],
    "residual_is_zero": bool(residual == sp.zeros(3, 1)),
    "sphere_z_plus_to_x_plus_transition": {
        "map": ["y", "sqrt(1 - x^2 - y^2)"],
        "determinant": str(sphere_transition_det),
        "nonzero_on_overlap_condition": "x > 0 and 1 - x^2 - y^2 > 0",
    },
}
CHECKS["reparametrization"] = reparam_check
save_check(reparam_check, "reparametrization-cross-product.json")

display_artifact(reparam_png, width=880)
reparam_check

## Smooth maps between surfaces: local diffeomorphism is not global injectivity

A smooth map between surfaces is tested in coordinates. The wrapping map from the `yz`-plane to the unit cylinder sends `(0, y, z)` to `(cos y, sin y, z)`. It is not one-to-one: increasing `y` by `2*pi` returns to the same cylinder point. But near any selected `y` value, one can choose an angle chart on the cylinder, and the coordinate representative has Jacobian determinant `1`.

In the artifact, the flat sheet is drawn beside the cylinder only to keep the domain and image visible at the same time. The colored horizontal lines on the sheet wrap to colored circles on the cylinder.

In [ ]:
y_vals = np.linspace(-4 * math.pi, 4 * math.pi, 180)
z_vals = np.linspace(-1.4, 1.4, 24)
Yp, Zp = np.meshgrid(y_vals / (2 * math.pi), z_vals, indexing="ij")
Xp = -3.0 * np.ones_like(Yp)
angle = np.linspace(0, 2 * math.pi, 130)
z_grid = np.linspace(-1.4, 1.4, 40)
Tc, Zc = np.meshgrid(angle, z_grid, indexing="ij")
Xc, Yc = np.cos(Tc), np.sin(Tc)

fig = go.Figure()
fig.add_surface(x=Xp, y=Yp, z=Zp, opacity=0.42, colorscale="Greens", showscale=False, name="domain sheet")
fig.add_surface(x=Xc, y=Yc, z=Zc, opacity=0.72, colorscale="Blues", showscale=False, name="unit cylinder")
for z0, color in [(-0.75, "#c44e52"), (0.45, "#2a9d8f")]:
    fig.add_trace(go.Scatter3d(x=-3 * np.ones_like(y_vals), y=y_vals / (2 * math.pi), z=z0 * np.ones_like(y_vals), mode="lines", line={"color": color, "width": 6}, name=f"plane line z={z0}"))
    fig.add_trace(go.Scatter3d(x=np.cos(angle), y=np.sin(angle), z=z0 * np.ones_like(angle), mode="lines", line={"color": color, "width": 6}, name=f"wrapped circle z={z0}"))
fig.update_layout(
    title="Smooth map: the plane wraps locally but not globally onto the cylinder",
    scene={"aspectmode": "data", "xaxis_title": "x", "yaxis_title": "scaled y / cylinder y", "zaxis_title": "z"},
    margin={"l": 0, "r": 0, "t": 45, "b": 0},
)
wrap_html = save_plotly(fig, "plane-to-cylinder-local-diffeomorphism.html")

local_map_u, local_map_v = sp.symbols("local_map_u local_map_v", real=True)
F = sp.Matrix([local_map_u, local_map_v])
J_F = F.jacobian([local_map_u, local_map_v])
wrap_check = {
    "coordinate_representative_on_one_angle_strip": ["u", "v"],
    "local_jacobian": [[int(J_F[i, j]) for j in range(2)] for i in range(2)],
    "local_determinant": int(J_F.det()),
    "noninjective_witness_domain_points": [[0, 0, 0], [0, float(2 * math.pi), 0]],
    "noninjective_witness_same_image_distance": float(np.linalg.norm(np.array([1, 0, 0]) - np.array([math.cos(2 * math.pi), math.sin(2 * math.pi), 0]))),
}
CHECKS["smooth_map_plane_to_cylinder"] = wrap_check
save_check(wrap_check, "smooth-map-local-diffeomorphism.json")

fig2, ax = plt.subplots(figsize=(8.8, 3.8))
ax.axis("off")
ax.add_patch(plt.Rectangle((0.05, 0.55), 0.26, 0.22, facecolor="white", edgecolor="#9aa4b2", linewidth=1.4))
ax.add_patch(plt.Rectangle((0.69, 0.55), 0.26, 0.22, facecolor="white", edgecolor="#9aa4b2", linewidth=1.4))
ax.add_patch(plt.Rectangle((0.37, 0.12), 0.26, 0.22, facecolor="white", edgecolor="#9aa4b2", linewidth=1.4))
ax.text(0.18, 0.66, "T_p S\nbasis sigma_u, sigma_v", ha="center", va="center", color=PALETTE["ink"], fontsize=10)
ax.text(0.82, 0.66, "T_f(p) S_tilde\nbasis sigma_tilde_u, sigma_tilde_v", ha="center", va="center", color=PALETTE["ink"], fontsize=10)
ax.text(0.50, 0.23, "coordinate map\n(alpha(u,v), beta(u,v))", ha="center", va="center", color=PALETTE["ink"], fontsize=10)
ax.annotate("D_p f", xy=(0.69, 0.66), xytext=(0.31, 0.66), arrowprops={"arrowstyle": "->", "lw": 1.8, "color": PALETTE["blue"]}, color=PALETTE["blue"], ha="center")
ax.annotate("Jacobian", xy=(0.63, 0.34), xytext=(0.69, 0.55), arrowprops={"arrowstyle": "->", "lw": 1.2, "color": PALETTE["teal"]}, color=PALETTE["teal"])
ax.annotate("coordinate velocity", xy=(0.37, 0.34), xytext=(0.25, 0.55), arrowprops={"arrowstyle": "->", "lw": 1.2, "color": PALETTE["teal"]}, color=PALETTE["teal"])
ax.text(0.5, 0.02, "The derivative is geometric; the matrix is its coordinate shadow.", ha="center", color=PALETTE["ink"], fontsize=10)
derivative_png = save_png(fig2, "derivative-jacobian-proof-state.png")

display_artifact(wrap_html, width="100%", height=520)
display_artifact(derivative_png, width=880)
wrap_check

## Tangent planes from parameter curves

A tangent vector to a surface is a velocity of a curve lying on the surface. In coordinates, a curve has the form `gamma(t) = sigma(u(t), v(t))`, so the chain rule says

`gamma_dot = sigma_u u_dot + sigma_v v_dot`.

The artifact uses the saddle graph `sigma(u,v) = (u, v, u^2 - v^2)`. The red and green surface curves are the two parameter curves through the chosen point. Their tangent vectors span the translucent tangent plane, and the blue arrow is perpendicular to that span.

In [ ]:
def saddle_point(u_value, v_value):
    return np.array([u_value, v_value, u_value**2 - v_value**2], dtype=float)

u0, v0 = 0.70, -0.40
p0 = saddle_point(u0, v0)
su = np.array([1.0, 0.0, 2.0 * u0])
sv = np.array([0.0, 1.0, -2.0 * v0])
normal_raw = np.cross(su, sv)
normal = normal_raw / np.linalg.norm(normal_raw)

ug = np.linspace(-1.15, 1.15, 70)
vg = np.linspace(-1.0, 1.0, 65)
U, V = np.meshgrid(ug, vg, indexing="ij")
X, Y, Z = U, V, U**2 - V**2
plane_a = np.linspace(-0.55, 0.55, 8)
plane_b = np.linspace(-0.55, 0.55, 8)
A, B = np.meshgrid(plane_a, plane_b, indexing="ij")
Plane = p0 + A[..., None] * su + B[..., None] * sv
curve_u = np.array([saddle_point(t, v0) for t in ug])
curve_v = np.array([saddle_point(u0, t) for t in vg])

fig = go.Figure()
fig.add_surface(x=X, y=Y, z=Z, opacity=0.68, colorscale="Viridis", showscale=False, name="saddle patch")
fig.add_surface(x=Plane[..., 0], y=Plane[..., 1], z=Plane[..., 2], opacity=0.38, colorscale=[[0, "#f3d37a"], [1, "#f3d37a"]], showscale=False, name="tangent plane")
fig.add_trace(go.Scatter3d(x=curve_u[:, 0], y=curve_u[:, 1], z=curve_u[:, 2], mode="lines", line={"color": "#c44e52", "width": 7}, name="v constant parameter curve"))
fig.add_trace(go.Scatter3d(x=curve_v[:, 0], y=curve_v[:, 1], z=curve_v[:, 2], mode="lines", line={"color": "#2a9d8f", "width": 7}, name="u constant parameter curve"))
fig.add_trace(go.Scatter3d(x=[p0[0]], y=[p0[1]], z=[p0[2]], mode="markers", marker={"size": 5, "color": "black"}, name="chosen point"))
fig.add_trace(segment_trace(p0, p0 + 0.45 * su, "sigma_u", "#c44e52"))
fig.add_trace(segment_trace(p0, p0 + 0.45 * sv, "sigma_v", "#2a9d8f"))
fig.add_trace(cone_trace(p0, 0.55 * normal, "unit normal", "#2f6fbb", sizeref=0.18))
fig.update_layout(
    title="Tangent plane from parameter curves on a saddle patch",
    scene={"aspectmode": "data", "xaxis_title": "x", "yaxis_title": "y", "zaxis_title": "z"},
    margin={"l": 0, "r": 0, "t": 45, "b": 0},
)
tangent_html = save_plotly(fig, "tangent-plane-from-parameter-curves.html")

lam, mu = 0.6, -0.25
h = 1e-6
finite_difference = (saddle_point(u0 + lam * h, v0 + mu * h) - saddle_point(u0 - lam * h, v0 - mu * h)) / (2 * h)
predicted_velocity = lam * su + mu * sv
chain_rule_residual = float(np.linalg.norm(finite_difference - predicted_velocity))
tangent_check = {
    "point": p0.tolist(),
    "sigma_u": su.tolist(),
    "sigma_v": sv.tolist(),
    "normal_unit_length": float(np.linalg.norm(normal)),
    "normal_dot_sigma_u": float(np.dot(normal, su)),
    "normal_dot_sigma_v": float(np.dot(normal, sv)),
    "chain_rule_residual": chain_rule_residual,
    "tangent_plane_rank": int(np.linalg.matrix_rank(np.column_stack([su, sv]))),
}
CHECKS["tangent_plane"] = tangent_check
save_check(tangent_check, "tangent-plane-chain-rule.json")

display_artifact(tangent_html, width="100%", height=560)
tangent_check

## Normals and oriented angles

For a regular patch, `sigma_u x sigma_v` is perpendicular to every tangent vector in the plane it spans. Normalizing gives the standard unit normal of that patch. For the latitude-longitude sphere patch used in the source span, the standard normal points inward on its open domain. That sign matters: changing the normal reverses oriented angles in the tangent plane.

In the artifact, the cones show the inward normals at sample points on the sphere. The numerical check at `p = (1,0,0)` compares the oriented angle from `v = (0,1,0)` to `w = (0,0,1)` using inward and outward normals.

In [ ]:
theta_vals = np.linspace(-1.15, 1.15, 70)
phi_vals = np.linspace(0, 2 * math.pi, 100)
TH, PH = np.meshgrid(theta_vals, phi_vals, indexing="ij")
Xs = np.cos(TH) * np.cos(PH)
Ys = np.cos(TH) * np.sin(PH)
Zs = np.sin(TH)

fig = go.Figure()
fig.add_surface(x=Xs, y=Ys, z=Zs, opacity=0.68, colorscale="Blues", showscale=False, name="sphere patch")
for th in [-0.8, 0.0, 0.8]:
    for ph in [0.0, math.pi / 2, math.pi, 3 * math.pi / 2]:
        p = np.array([math.cos(th) * math.cos(ph), math.cos(th) * math.sin(ph), math.sin(th)])
        fig.add_trace(cone_trace(p, -0.23 * p, "inward normal", "#c44e52", sizeref=0.09))
fig.update_layout(
    title="Standard normals for the latitude-longitude sphere patch point inward",
    scene={"aspectmode": "data", "xaxis_title": "x", "yaxis_title": "y", "zaxis_title": "z"},
    margin={"l": 0, "r": 0, "t": 45, "b": 0},
    showlegend=False,
)
sphere_normals_html = save_plotly(fig, "sphere-standard-normal-field.html")

p = np.array([1.0, 0.0, 0.0])
sigma_theta = np.array([0.0, 0.0, 1.0])
sigma_phi = np.array([0.0, 1.0, 0.0])
standard_normal = np.cross(sigma_theta, sigma_phi)
standard_normal = standard_normal / np.linalg.norm(standard_normal)
outward_normal = p
v_vec = np.array([0.0, 1.0, 0.0])
w_vec = np.array([0.0, 0.0, 1.0])

def signed_angle(v1, v2, normal_vector):
    return float(math.atan2(np.dot(normal_vector, np.cross(v1, v2)), np.dot(v1, v2)))

normal_check = {
    "standard_normal_at_(1,0,0)": standard_normal.tolist(),
    "standard_normal_is_inward": bool(np.allclose(standard_normal, -p)),
    "normal_unit_error": float(abs(np.linalg.norm(standard_normal) - 1.0)),
    "normal_dot_sigma_theta": float(np.dot(standard_normal, sigma_theta)),
    "normal_dot_sigma_phi": float(np.dot(standard_normal, sigma_phi)),
    "oriented_angle_inward_radians": signed_angle(v_vec, w_vec, standard_normal),
    "oriented_angle_outward_radians": signed_angle(v_vec, w_vec, outward_normal),
}
CHECKS["sphere_normals"] = normal_check
save_check(normal_check, "normals-oriented-angle.json")

display_artifact(sphere_normals_html, width="100%", height=540)
normal_check

## Orientability and the Mobius obstruction

An orientable atlas can keep every transition determinant positive, which lets the standard normal from any chart agree with the same global choice. The Mobius band blocks that plan. Locally it has regular patches, but one trip around the median circle returns to the same point with the transported standard normal reversed.

The artifact displays the twisted strip and sampled normals along the median circle. The check has two independent signs of the obstruction: the transition map between the two natural Mobius charts has one positive and one negative determinant branch, and the median normal endpoint dot product is `-1` in the limiting calculation.

In [ ]:
def mobius_point(t_value, theta_value):
    return np.array([
        (1 - t_value * math.sin(theta_value / 2)) * math.cos(theta_value),
        (1 - t_value * math.sin(theta_value / 2)) * math.sin(theta_value),
        t_value * math.cos(theta_value / 2),
    ], dtype=float)

def mobius_normal_median(theta_value):
    return np.array([
        -math.cos(theta_value) * math.cos(theta_value / 2),
        -math.sin(theta_value) * math.cos(theta_value / 2),
        -math.sin(theta_value / 2),
    ], dtype=float)

t_vals = np.linspace(-0.48, 0.48, 36)
th_vals = np.linspace(0, 2 * math.pi, 150)
TT, HH = np.meshgrid(t_vals, th_vals, indexing="ij")
Xm = (1 - TT * np.sin(HH / 2)) * np.cos(HH)
Ym = (1 - TT * np.sin(HH / 2)) * np.sin(HH)
Zm = TT * np.cos(HH / 2)

fig = go.Figure()
fig.add_surface(x=Xm, y=Ym, z=Zm, surfacecolor=TT, colorscale="Viridis", opacity=0.78, showscale=False, name="Mobius band")
median_theta = np.linspace(0, 2 * math.pi, 220)
median = np.array([mobius_point(0.0, th) for th in median_theta])
fig.add_trace(go.Scatter3d(x=median[:, 0], y=median[:, 1], z=median[:, 2], mode="lines", line={"color": "black", "width": 7}, name="median circle"))
for th in np.linspace(0.18, 2 * math.pi - 0.18, 13):
    p = mobius_point(0.0, th)
    n = mobius_normal_median(th)
    fig.add_trace(cone_trace(p, 0.22 * n, "transported median normal", "#c44e52", sizeref=0.08))
start_p = mobius_point(0.0, 0.03)
end_p = mobius_point(0.0, 2 * math.pi - 0.03)
fig.add_trace(segment_trace(start_p, start_p + 0.45 * mobius_normal_median(0.03), "normal near theta=0", "#2f6fbb", width=8))
fig.add_trace(segment_trace(end_p, end_p + 0.45 * mobius_normal_median(2 * math.pi - 0.03), "normal near theta=2*pi", "#d59f0f", width=8))
fig.update_layout(
    title="Mobius band: local normals reverse after one circuit",
    scene={"aspectmode": "data", "xaxis_title": "x", "yaxis_title": "y", "zaxis_title": "z"},
    margin={"l": 0, "r": 0, "t": 45, "b": 0},
)
mobius_html = save_plotly(fig, "mobius-normal-obstruction.html")

t_sym, th_sym = sp.symbols("t_tilde theta_tilde", real=True)
positive_branch = sp.Matrix([t_sym, th_sym])
negative_branch = sp.Matrix([-t_sym, th_sym + 2 * sp.pi])
positive_det = sp.simplify(positive_branch.jacobian([t_sym, th_sym]).det())
negative_det = sp.simplify(negative_branch.jacobian([t_sym, th_sym]).det())
N_start = np.array([-1.0, 0.0, 0.0])
N_end = np.array([1.0, 0.0, 0.0])

fig2, ax = plt.subplots(figsize=(8.8, 3.8))
ax.set_title("Mobius transition branches have incompatible orientation signs", color=PALETTE["ink"])
ax.set_xlim(-math.pi - 0.3, math.pi + 0.3)
ax.set_ylim(-0.65, 0.65)
ax.set_xlabel("theta_tilde in the second chart")
ax.set_ylabel("t_tilde")
ax.axvspan(0, math.pi, color="#dbeafe", alpha=0.9, label="Phi(t,theta)=(t,theta), det=+1")
ax.axvspan(-math.pi, 0, color="#fee2e2", alpha=0.9, label="Phi(t,theta)=(-t,theta+2*pi), det=-1")
ax.axhline(0, color="#606b78", linewidth=1)
ax.axvline(0, color="#606b78", linewidth=1)
ax.legend(loc="upper center", fontsize=8)
for spine in ax.spines.values():
    spine.set_color("#b6c0ca")
mobius_transition_png = save_png(fig2, "mobius-transition-determinant-signs.png")

mobius_check = {
    "transition_determinants": {"positive_overlap_branch": int(positive_det), "negative_overlap_branch": int(negative_det)},
    "contains_negative_transition_determinant": bool(negative_det < 0),
    "median_normal_limit_theta_down_0": N_start.tolist(),
    "median_normal_limit_theta_up_2pi": N_end.tolist(),
    "endpoint_normal_dot_product": float(np.dot(N_start, N_end)),
    "obstruction_detected": bool(np.dot(N_start, N_end) < -0.999 and negative_det < 0),
}
CHECKS["mobius_obstruction"] = mobius_check
save_check(mobius_check, "mobius-orientability-obstruction.json")

display_artifact(mobius_html, width="100%", height=560)
display_artifact(mobius_transition_png, width=880)
mobius_check

## Applied Lab: Patch doctor for proposed surfaces

The lab is a quick diagnostic workflow. Given a proposed patch, compute `||sigma_u x sigma_v||` on the intended parameter region and read it as a regularity meter. A small value does not prove singularity by itself, but an exact zero or a forced zero curve tells you that the formula cannot be used as an allowable patch there.

Use the table as a checklist before trusting a parametrization: What is the declared domain? Is the cross product nonzero on that domain? Is a seam or excluded point a coordinate artifact, or is it a real surface obstruction?

In [ ]:
def min_cross_norm_for_patch(su_func, sv_func, samples):
    values = []
    for params in samples:
        values.append(np.linalg.norm(np.cross(su_func(*params), sv_func(*params))))
    return float(np.min(values))

uv_grid = [(u_value, v_value) for u_value in np.linspace(-1, 1, 31) for v_value in np.linspace(-1, 1, 31)]
theta_phi_open = [(th, ph) for th in np.linspace(-1.2, 1.2, 31) for ph in np.linspace(0, 2 * math.pi, 31)]
theta_phi_closed = [(th, ph) for th in np.linspace(-math.pi / 2, math.pi / 2, 31) for ph in np.linspace(0, 2 * math.pi, 31)]
bad_grid = [(u_value, v_value) for u_value in np.linspace(-1, 1, 31) for v_value in np.linspace(-1, 1, 31)]
rtheta_grid = [(r, th) for r in np.linspace(0.2, 1.4, 30) for th in np.linspace(0, 2 * math.pi, 30)]

rows = []
rows.append({
    "candidate": "graph sigma(u,v)=(u,v,u^2-v^2)",
    "declared_domain": "rectangle sample in R^2",
    "min_cross_norm": min_cross_norm_for_patch(lambda u, v: np.array([1, 0, 2 * u]), lambda u, v: np.array([0, 1, -2 * v]), uv_grid),
    "diagnosis": "regular on every sampled point; graph patches have a built-in nonzero vertical cross component",
})
rows.append({
    "candidate": "sphere latitude-longitude open band",
    "declared_domain": "-1.2 <= theta <= 1.2 sample, away from poles",
    "min_cross_norm": min_cross_norm_for_patch(
        lambda th, ph: np.array([-math.sin(th) * math.cos(ph), -math.sin(th) * math.sin(ph), math.cos(th)]),
        lambda th, ph: np.array([-math.cos(th) * math.sin(ph), math.cos(th) * math.cos(ph), 0]),
        theta_phi_open,
    ),
    "diagnosis": "regular while cos(theta) stays away from zero",
})
rows.append({
    "candidate": "sphere latitude-longitude including poles",
    "declared_domain": "closed theta range includes +/- pi/2",
    "min_cross_norm": min_cross_norm_for_patch(
        lambda th, ph: np.array([-math.sin(th) * math.cos(ph), -math.sin(th) * math.sin(ph), math.cos(th)]),
        lambda th, ph: np.array([-math.cos(th) * math.sin(ph), math.cos(th) * math.cos(ph), 0]),
        theta_phi_closed,
    ),
    "diagnosis": "not an allowable single patch across the poles; the cross product vanishes",
})
rows.append({
    "candidate": "bad patch sigma(u,v)=(u,v^2,v^3)",
    "declared_domain": "rectangle sample in R^2",
    "min_cross_norm": min_cross_norm_for_patch(lambda u, v: np.array([1, 0, 0]), lambda u, v: np.array([0, 2 * v, 3 * v**2]), bad_grid),
    "diagnosis": "singular along v=0; the two parameter directions fail to span a plane",
})
rows.append({
    "candidate": "punctured half-cone polar patch",
    "declared_domain": "r >= 0.2 sample; vertex excluded",
    "min_cross_norm": min_cross_norm_for_patch(
        lambda r, th: np.array([math.cos(th), math.sin(th), 1.0]),
        lambda r, th: np.array([-r * math.sin(th), r * math.cos(th), 0.0]),
        rtheta_grid,
    ),
    "diagnosis": "regular on the punctured half-cone, but the vertex is not part of this surface patch",
})

lab_table = pd.DataFrame(rows)
lab_path = TABLES_DIR / "applied-lab-regularity-table.csv"
lab_table.to_csv(lab_path, index=False)
register(lab_path)
CHECKS["applied_lab"] = {
    "row_count": int(len(lab_table)),
    "regular_rows_with_positive_minimum": int((lab_table["min_cross_norm"] > 1e-6).sum()),
    "singular_rows_detected": lab_table.loc[lab_table["min_cross_norm"] <= 1e-6, "candidate"].tolist(),
    "table_path": book_relative(lab_path),
}
save_check(CHECKS["applied_lab"], "applied-lab-regularity-check.json")
lab_table

## final_sanity checks

This final cell checks the core identities and the durability of the artifact set. It deliberately tests both mathematical promises and file-level promises: atlas coverage, nonzero Jacobian determinants where needed, exact reparametrization identity, tangent-plane chain rule, normal orthogonality, Mobius sign obstruction, artifact existence, and nonblank PNG files.

In [ ]:
assert CHECKS["cylinder_atlas"]["minimum_sample_coverage"] >= 1
assert all(branch["determinant"] > 0 for branch in CHECKS["cylinder_atlas"]["cylinder_transition_branches"])
assert CHECKS["reparametrization"]["residual_is_zero"]
assert CHECKS["smooth_map_plane_to_cylinder"]["local_determinant"] == 1
assert CHECKS["smooth_map_plane_to_cylinder"]["noninjective_witness_same_image_distance"] < 1e-12
assert CHECKS["tangent_plane"]["tangent_plane_rank"] == 2
assert abs(CHECKS["tangent_plane"]["normal_unit_length"] - 1.0) < 1e-12
assert abs(CHECKS["tangent_plane"]["normal_dot_sigma_u"]) < 1e-12
assert abs(CHECKS["tangent_plane"]["normal_dot_sigma_v"]) < 1e-12
assert CHECKS["tangent_plane"]["chain_rule_residual"] < 1e-8
assert CHECKS["sphere_normals"]["standard_normal_is_inward"]
assert abs(CHECKS["sphere_normals"]["oriented_angle_inward_radians"] + math.pi / 2) < 1e-12
assert abs(CHECKS["sphere_normals"]["oriented_angle_outward_radians"] - math.pi / 2) < 1e-12
assert CHECKS["mobius_obstruction"]["contains_negative_transition_determinant"]
assert CHECKS["mobius_obstruction"]["endpoint_normal_dot_product"] < -0.999
assert CHECKS["mobius_obstruction"]["obstruction_detected"]
assert CHECKS["applied_lab"]["row_count"] >= 5
assert any("bad patch" in name for name in CHECKS["applied_lab"]["singular_rows_detected"])

artifact_summary = []
for path in ARTIFACT_PATHS:
    min_bytes = 128 if path.suffix.lower() in {".json", ".csv"} else 512
    assert_artifact(path, min_bytes=min_bytes)
    entry = {"path": book_relative(path), "bytes": int(path.stat().st_size)}
    if path.suffix.lower() == ".png":
        entry["image_stats"] = require_nonblank_png(path)
    artifact_summary.append(entry)

final_sanity = {
    "unit": UNIT,
    "source_span": source_span,
    "artifact_count_before_final_json": len(artifact_summary),
    "artifacts": artifact_summary,
    "checks": CHECKS,
}
final_sanity_path = save_check(final_sanity, "final-sanity.json")
assert_artifact(final_sanity_path, min_bytes=1024)
final_sanity["final_sanity_path"] = book_relative(final_sanity_path)
final_sanity

## Takeaways

A surface is not the same thing as one parametrized picture. The chapter's working object is an atlas: patches, domains, and transition maps together.

Regularity is the local rank-two condition `sigma_u x sigma_v != 0`. Reparametrization can change the coordinate basis, but the cross product only changes by the determinant of the coordinate change, so the tangent plane is well defined.

Smooth maps and derivatives are geometric, while their matrices are coordinate representatives. The plane can wrap smoothly and locally diffeomorphically around the cylinder without being globally one-to-one.

Normals remember orientation. On orientable surfaces the transition determinants can be kept positive so normals glue consistently. On the Mobius band one overlap branch reverses determinant sign, and the transported median normal returns as its negative.